In [414]:
import torch
import torch.nn.functional as F

In [415]:
words = open("names.txt", "r").read().splitlines()

words[:3]

['emma', 'olivia', 'ava']

In [416]:
chars = sorted(list(set("".join(words))))

In [417]:
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi["."] = 0

itos = {i: s for s, i in stoi.items()}

In [418]:
block_size = 3 
X, Y = [], [] 

for w in words[:5]:
    print(w)
    context = [0] * block_size 

    for ch in w + ".":
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print("".join(itos[i] for i in context), "--->", itos[ix])
        context = context[1:] + [ix]   

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [419]:
X.shape, Y.shape

(torch.Size([32, 3]), torch.Size([32]))

In [420]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1]])

In [421]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [426]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)

In [427]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [428]:
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)

In [430]:
parameters = [W1, b1, W2, b2]

for p in parameters: 
    p.requires_grad = True

sum(p.nelement() for p in parameters)

3427

In [456]:
for _ in range(10000):
    # forward pass 
    emb = C[X]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2 
    #counts = logits.exp()
    #probs = counts / counts.sum(1, keepdim=True)
    #loss = -probs[torch.arange(32), Y].log().mean()
    loss = F.cross_entropy(logits, Y)
    print(loss.item())

    # backward pass 
    for p in parameters: 
        p.grad = None 
    loss.backward()

    # update 
    lr = 0.1 
    for p in parameters: 
        p.data -= lr * p.grad

loss 

0.2571112811565399
0.2571098804473877
0.25710856914520264
0.2571072280406952
0.25710588693618774
0.2571045458316803
0.25710320472717285
0.2571018636226654
0.25710055232048035
0.2570992112159729
0.25709787011146545
0.2570964992046356
0.2570951581001282
0.2570938169956207
0.25709250569343567
0.2570911645889282
0.2570898234844208
0.2570885121822357
0.25708717107772827
0.2570858299732208
0.2570844888687134
0.25708314776420593
0.2570818364620209
0.2570805251598358
0.25707918405532837
0.2570778429508209
0.25707653164863586
0.2570751905441284
0.2570738196372986
0.2570725083351135
0.25707119703292847
0.2570698857307434
0.2570685148239136
0.2570672035217285
0.25706589221954346
0.2570645809173584
0.25706326961517334
0.2570618987083435
0.25706058740615845
0.2570592761039734
0.25705796480178833
0.2570566236972809
0.2570553421974182
0.25705400109291077
0.2570526897907257
0.25705140829086304
0.2570500671863556
0.25704875588417053
0.2570474147796631
0.257046103477478
0.2570447325706482
0.257043480873

tensor(0.2530, grad_fn=<NllLossBackward0>)